# 📊 Informe Completo: Evaluación y Comparación de Modelos de Embeddings

## 🎯 Objetivo del Análisis
El objetivo de este informe es evaluar exhaustivamente el rendimiento de varios modelos de embeddings, analizando no solo su capacidad de recuperación (**MRR**, **Top-K Accuracy**) sino también su eficiencia computacional (**Tiempo de Embedding**, **Consumo de RAM**) y el impacto de diferentes estrategias de fragmentación (**Chunk Size** y **Overlap**).

Este análisis te permitirá seleccionar el modelo óptimo que ofrezca el mejor equilibrio entre precisión semántica y latencia/costo de infraestructura.


In [13]:
# Importar las librerías necesarias
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

# Configuración visual limpia por defecto para Plotly
import plotly.io as pio
pio.templates.default = 'plotly_white'

print('✅ Librerías importadas correctamente. Listo para analizar.')


✅ Librerías importadas correctamente. Listo para analizar.


## 1. Carga y Exploración de Datos

Cargamos los datos generados por tu script de evaluación. El sistema leerá dinámicamente tanto los resultados individuales como los promedios calculados por modelo.


In [ ]:
# Ruta del archivo (puedes cambiarla cuando tengas el archivo definitivo más extenso)
FILE_PATH = 'embedding_comparison_results.xlsx'

try:
    # Cargar ambas hojas del libro de Excel
    df_completos = pd.read_excel(FILE_PATH, sheet_name='Resultados_Completos')
    df_promedios = pd.read_excel(FILE_PATH, sheet_name='Resumen_Promedios')
    print(f'¡Datos cargados con éxito! Registros completos: {len(df_completos)} | Modelos evaluados: {len(df_promedios)}')
    display(df_promedios.head())
except Exception as e:
    print(f'Error {e}')


✅ ¡Datos cargados con éxito! Registros completos: 45 | Modelos evaluados: 5


,Modelo,Chunk Size,Overlap,Total Tokens,RAM (MB),T. Chunking (s),T. Embedding (s),MRR,Top_1_Acc,Top_3_Acc,LLM_Judge,Eficiencia (MRR / T.Emb)
0,intfloat/multilingual-e5-base,1000,166.666667,5396.666667,3739.046667,0.000522,4.360000,0.698856,0.549733,0.796478,0,0.000000
1,intfloat/multilingual-e5-small,1000,166.666667,5396.666667,3595.786667,0.000511,2.081111,0.658600,0.511333,0.767956,0,0.111111
2,sentence-transformers/LaBSE,1000,166.666667,5090.111111,3888.171111,0.000433,3.444444,0.727767,0.574922,0.832278,0,0.000000
3,sentence-transformers/distiluse-base-multiling...,1000,166.666667,5872.222222,3865.418889,0.000500,1.771111,0.625333,0.452556,0.775844,0,0.333333
4,sentence-transformers/paraphrase-multilingual-...,1000,166.666667,5396.666667,3542.924444,0.000500,0.907778,0.594267,0.449778,0.680922,0,0.888889


In [15]:
# Mapeo con dimensiones reales: MiniLM (384), E5-Small (384), E5-Base (768), 
# Distiluse (512), Jina-v3 (1024), Jina-v2-ES (768), LaBSE (768).
nombres_cortos = {
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"   : "Mul-384-MiniLM-L12",
    "intfloat/multilingual-e5-small"                                : "Mul-384-E5-Small",
    "intfloat/multilingual-e5-base"                                 : "Mul-768-E5-Base",
    "sentence-transformers/distiluse-base-multilingual-cased-v1"    : "Mul-512-Distiluse-v1",
    "jinaai/jina-embeddings-v3"                                     : "Mul-1024-Jina-v3",
    "jinaai/jina-embeddings-v2-base-es"                             : "ESP-768-Jina-v2",
    "sentence-transformers/LaBSE"                                   : "Mul-768-LaBSE"
}

# Aplicar el reemplazo en tus DataFrames
df_completos['Modelo'] = df_completos['Modelo'].replace(nombres_cortos)
df_promedios['Modelo'] = df_promedios['Modelo'].replace(nombres_cortos)

print("✅ Nombres estructurados por Idioma-Dimensión-Modelo listos.")

✅ Nombres estructurados por Idioma-Dimensión-Modelo listos.


## 🏆 2. Análisis de Rendimiento de Recuperación (Precisión)

La métrica reina para evaluar sistemas RAG o de búsqueda semántica es el **MRR (Mean Reciprocal Rank)**, ya que penaliza fuertemente si la respuesta correcta decae en posiciones bajas. Paralelamente, comparamos la certeza en los primeros puestos (**Top-1 Acc** y **Top-3 Acc** o el Top-K configurado).


In [16]:
# 1. MRR Promedio por Modelo
fig_mrr = px.bar(
    df_promedios.sort_values('MRR', ascending=True), 
    x='MRR', 
    y='Modelo', 
    orientation='h',
    title='<b>Métricas Globales: MRR Promedio por Modelo</b>',
    text='MRR',
    color='MRR',
    color_continuous_scale='Viridis'
)
fig_mrr.update_traces(texttemplate='%{text:.3f}', textposition='outside')
fig_mrr.update_layout(height=450, xaxis_title='Mean Reciprocal Rank (MRR)', yaxis_title='')
fig_mrr.show()

# 2. Comparativa de Precisión Histográfica (Top-1 vs Top-K)
fig_acc = go.Figure()
fig_acc.add_trace(go.Bar(
    x=df_promedios['Modelo'], y=df_promedios['Top_1_Acc'],
    name='Top 1 Accuracy', marker_color='#2b5c8f'
))

# Buscar dinámicamente la otra columna de precisión Top_K que se haya generado
col_top_k = [c for c in df_promedios.columns if 'Top_' in c and c != 'Top_1_Acc']
label_top_k = col_top_k[0] if col_top_k else 'Top_3_Acc'

fig_acc.add_trace(go.Bar(
    x=df_promedios['Modelo'], y=df_promedios[label_top_k],
    name=label_top_k.replace('_', ' '), marker_color='#4682B4'
))

fig_acc.update_layout(
    title='<b>Acierto Semántico: Top-1 vs Contexto Ampliado</b>',
    barmode='group', bargap=0.15, height=450,
    yaxis=dict(title='Ratio de Acierto (Accuracy)'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig_acc.show()


## ⏱️ 3. Análisis de Eficiencia, Tiempos y Consumo de Recursos

Un modelo con un MRR perfecto puede ser inviable si introduce una latencia alta en producción o desborda la memoria RAM disponible. Buscamos el punto óptimo (*Sweet Spot*).


In [17]:
# 1. Gráfico de Dispersión: Precisión vs Velocidad
fig_tradeoff = px.scatter(
    df_promedios, 
    x='T. Embedding (s)', 
    y='MRR', 
    color='Modelo',
    size='Total Tokens',
    hover_data=['RAM (MB)'],
    title='<b>Análisis Coste-Beneficio: Precisión (MRR) vs Tiempo de Cómputo</b><br><sup>El tamaño de la burbuja representa los Tokens totales indexados. Lo ideal es arriba a la izquierda.</sup>',
    labels={'T. Embedding (s)': 'Tiempo de Cómputo Promedio (Segundos)', 'MRR': 'Calidad de Recuperación (MRR)'}
)
fig_tradeoff.update_traces(marker=dict(line=dict(width=1, color='DarkSlateGrey')))
fig_tradeoff.update_layout(height=550)
fig_tradeoff.show()

# 2. Variabilidad del Tiempo (Boxplot con los datos completos)
fig_box = px.box(
    df_completos, 
    x='Modelo', 
    y='T. Embedding (s)', 
    color='Modelo',
    title='<b>Estabilidad del Modelo: Variabilidad del Tiempo de Respuesta</b>',
    points='all'
)
fig_box.update_layout(height=450, showlegend=False, xaxis_title='')
fig_box.show()


## 🧩 4. Impacto de la Estrategia de Chunking (Segmentación)

Aquí analizamos cómo interactúan el tamaño de los fragmentos (`Chunk Size`) y su solapamiento (`Overlap`) para mejorar o empeorar el contexto capturado por los modelos.


In [18]:
# 1. Heatmap de impacto de segmentación global
try:
    pivot_chunking = df_completos.pivot_table(
        values='MRR', index='Chunk Size', columns='Overlap', aggfunc='mean'
    )
    fig_heatmap = px.imshow(
        pivot_chunking, text_auto='.3f', aspect='auto',
        color_continuous_scale='Blues',
        title='<b>Rendimiento de MRR según la anatomía del Chunking</b>'
    )
    fig_heatmap.update_layout(xaxis_title='Superposición (Overlap)', yaxis_title='Tamaño del Chunk (Caracteres)')
    fig_heatmap.show()
except Exception as e:
    print(f'Aviso sobre el Heatmap: {e} (Se requiere variedad de combinaciones Chunk/Overlap en los datos)')

# 2. Estructura Jerárquica: Modelo -> Chunk -> Overlap
fig_sunburst = px.sunburst(
    df_completos, 
    path=['Modelo', 'Chunk Size', 'Overlap'], 
    values='MRR', color='MRR',
    color_continuous_scale='RdYlGn',
    title='<b>Desglose Completo de Rendimiento Semántico</b>'
)
fig_sunburst.update_layout(height=650)
fig_sunburst.show()


## 💡 5. Conclusiones Estratégicas

Una vez ejecutados los gráficos con tu conjunto definitivo de datos, fíjate en estos tres pilares:
1. **Ganador absoluto en precisión:** ¿Qué modelo domina la parte alta del gráfico de MRR?
2. **El candidato de producción:** En el gráfico de dispersión, busca el modelo que esté más arriba a la izquierda. Ese te dará el menor coste en servidores y la menor latencia con una precisión óptima.
3. **Anatomía del texto:** El mapa de calor te desvelará si a tus modelos les favorecen los contextos concentrados (Chunks pequeños) o los flujos densos (Chunks grandes con alto overlap).
